In [ ]:
import pandas as pd
import sys
import os

sys.path.append(
    os.path.abspath("..")
)
from src.data_loader import load_data
from src.modeling import *


In [ ]:

df = load_data("../data/MachineLearningRating_v3.txt")
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str)
        df[col] = df[col].replace("nan", np.nan)
df["TransactionMonth"] = pd.to_datetime(df["TransactionMonth"], errors="coerce")

df["transaction_year"] = df["TransactionMonth"].dt.year
df["transaction_month"] = df["TransactionMonth"].dt.month

df = df.drop(columns=["TransactionMonth"])
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=["object"]).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")
df = df.dropna(subset=["TotalClaims", "TotalPremium"])


In [ ]:

claim_df = df[df["TotalClaims"] > 0]

X_train, X_test, y_train, y_test = split_data(claim_df, "TotalClaims")

reg_models = train_regression_models(X_train, y_train)

reg_results = []

for name, model in reg_models.items():
    preds = model.predict(X_test)
    rmse, r2 = evaluate_regression(y_test, preds)
    reg_results.append([name, rmse, r2])

pd.DataFrame(reg_results, columns=["Model", "RMSE", "R2"])


In [ ]:

df["has_claim"] = (df["TotalClaims"] > 0).astype(int)

X_train, X_test, y_train, y_test = split_data(df, "has_claim")

clf_models = train_classification_models(X_train, y_train)

clf_results = []

for name, model in clf_models.items():
    preds = model.predict(X_test)
    m = evaluate_classification(y_test, preds)
    clf_results.append([name, m["accuracy"], m["precision"], m["recall"], m["f1"]])

pd.DataFrame(clf_results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])